# Titanic Dataset Analysis
Comprehensive machine learning analysis of the Titanic dataset

# Titanic Dataset Analysis
Comprehensive machine learning analysis of the Titanic dataset

## 1. Data Imports
Loading essential libraries for data analysis, visualization, and machine learning

In [11]:
import pandas as pd, json, os, zipfile

## 2. Data Loading
Reading the Titanic dataset from CSV file

In [5]:
df=pd.read_csv('./data/train.csv')

## 3. Initial Exploration
Understanding the structure and shape of the dataset

In [10]:
print(df.shape, df.dtypes)

(891, 12) PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object


## 4. Missing Values
Identifying and analyzing missing data patterns

In [15]:
# LOADIND CSV FROM  a zip
with zipfile.ZipFile('./data/titanic.zip') as z:
    csv_files=[f for f in z.namelist() if f.endswith('.csv')]
    frame=[]
    for fname in csv_files:
        with z.open(fname) as f:
            frame.append(pd.read_csv(f))
            df_all=pd.concat(frame, ignore_index=True)

## 5. Data Head Inspection
Examining the first few rows of the dataset

In [17]:
df_all

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,893,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,894,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,895,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,896,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1722,887,0.0,2.0,"Montvila, Rev. Juozas",male,27.0,0.0,0.0,211536,13.00,NaN,S
1723,888,1.0,1.0,"Graham, Miss. Margaret Edith",female,19.0,0.0,0.0,112053,30.00,B42,S
1724,889,0.0,3.0,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1.0,2.0,W./C. 6607,23.45,NaN,S
1725,890,1.0,1.0,"Behr, Mr. Karl Howell",male,26.0,0.0,0.0,111369,30.00,C148,C


## 6. Data Info
Getting detailed information about data types and memory usage

usage: kaggle [-h] [-v] [-W]
              {competitions,c,datasets,d,kernels,k,models,m,files,f,config}
              ...
kaggle: error: unrecognized arguments: --unzip


In [24]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

train=pd.read_csv('./data/train.csv')


In [26]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [27]:
test=pd.read_csv('./titanic/test.csv')

In [28]:
test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [30]:
sub=pd.read_csv('./titanic/gender_submission.csv')

In [32]:
sub.head()

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


In [33]:
print('Train shape: ', train.shape)

Train shape:  (891, 12)


In [34]:
print('Test Shape: ', test.shape)

Test Shape:  (418, 11)


In [36]:
print(train.head(3))

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  


In [55]:
# Step 2 — Feature Engineering

def engineer_features(df):
    # title extraction from Name capture social status
    df['Title']=df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df['Title']=df['Title'].replace(['lady', 'Countess', 'Capt', 'Col', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    df['Title'] = df['Title'].replace({'Mlle':'Miss','Ms':'Miss','Master':'Mrs'})
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    df['Age']=df.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))
    df['Embarked']=df['Embarked']=df['Embarked'].fillna(df['Embarked'].mode())
    df['Fare']=df['Fare'].fillna(df['Fare'].median())
    df['Sex'] = LabelEncoder().fit_transform(df['Sex'])
    df['Embarked'] = LabelEncoder().fit_transform(df['Embarked'].astype(str))
    df['Title'] = LabelEncoder().fit_transform(df['Title'].astype(str))



    return df.head(10)
train = engineer_features(train)
test = engineer_features(test)
FEATURES = ['Pclass','Sex','Age','Fare','Embarked',
'FamilySize','IsAlone','Title']
X = train[FEATURES]
y = train['Survived']
X_test = test[FEATURES]
print("finally done")

finally done


In [57]:
X

,Pclass,Sex,Age,Fare,Embarked,FamilySize,IsAlone,Title
0,3,1,22.0,7.2500,2,2,0,1
1,1,0,38.0,71.2833,0,2,0,2
2,3,0,26.0,7.9250,2,1,1,0
3,1,0,35.0,53.1000,2,2,0,2
4,3,1,35.0,8.0500,2,1,1,1
5,3,1,30.0,8.4583,1,1,1,1
6,1,1,54.0,51.8625,2,1,1,1
7,3,1,2.0,21.0750,2,5,0,2
8,3,0,27.0,11.1333,2,3,0,2
9,2,0,14.0,30.0708,0,2,0,2


In [59]:
y

0    0
1    1
2    1
3    1
4    0
5    0
6    0
7    0
8    1
9    1
Name: Survived, dtype: int64

In [64]:
# TRAIN ->test ->Validate

cv=StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf=RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_split=5,
    random_state=43
)

rf_scores=cross_val_score(
    rf, X, y,
    cv=cv,
    scoring='accuracy'
)

In [66]:
rf_scores

array([0.5, 0.5, 1. , 0.5, 1. ])

In [68]:
gb = GradientBoostingClassifier(n_estimators=200, max_depth=4,
learning_rate=0.05, random_state=42)
gb_scores = cross_val_score(gb, X, y, cv=cv, scoring='accuracy')
print(f'GB CV: {gb_scores.mean():.4f} +/- {gb_scores.std():.4f}')
# Fit best model on full training data
best = gb if gb_scores.mean() > rf_scores.mean() else rf
best.fit(X, y)

GB CV: 1.0000 +/- 0.0000


,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.05
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",200
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are'friedman_mse' for the mean squared error with improvement score byFriedman, 'squared_error' for mean squared error. The default value of'friedman_mse' is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",4
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``,

In [69]:
# Generate predictions on test set
preds = best.predict(X_test)
# Build submission file — must match sample_submission.csv exactly
submission = pd.DataFrame({
'PassengerId': test['PassengerId'],
'Survived': preds
})
submission.to_csv('./titanic/my_submission.csv', index=False)
print(submission.head())
# Submit via Kaggle API
# kaggle competitions submit -c titanic \
# -f ./titanic/my_submission.csv -m 'GradientBoosting CV=0.831'
# Check your score on the leaderboard
# kaggle competitions submissions -c titanic

   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         1


In [72]:
!kaggle competitions submit -c titanic \
 -f ./titanic/my_submission.csv -m 'GradientBoosting CV=0.831'


403 Client Error: Forbidden for url: https://www.kaggle.com/api/v1/competitions/submission-url
